In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../datos/income_data.csv')

In [2]:
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


Ahora en la parte del preprocesing vamos a proceder a tratar los nulos y a trasformar las variables para poder tener un mejor modelo 

In [3]:
df = df.replace('?', np.nan)

In [4]:
df.isnull().sum()

age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64

In [5]:
for col in ['workclass', 'occupation', 'native-country']:
    df[col] = df[col].fillna('Unknown')

Durante el EDA se detectó que algunas variables utilizaban el carácter '?' para representar información desconocida.

Estos valores se han transformado a valores nulos y posteriormente se han imputado mediante la categoría "Unknown" para evitar sesgos y evitar aumentar la existencia de patrones inexistentes. 

Ahora procederemos a hacer transformacion de variables que hemos señalado durante el analisis univariable 

In [6]:
df.drop(columns=['education'], inplace=True)

se elimina education ya que la misma informacion esta represantada en educational num en formato numero mas comodo 

para simplificar la gran cantidad de paises que existen los agruparemos por continentes 

In [7]:
#IA

def agrupar_continente(pais):

    america_norte = [
        'United-States',
        'Canada'
    ]

    latinoamerica = [
        'Mexico',
        'Puerto-Rico',
        'Cuba',
        'Jamaica',
        'Dominican-Republic',
        'Guatemala',
        'El-Salvador',
        'Honduras',
        'Nicaragua',
        'Haiti',
        'Columbia',
        'Ecuador',
        'Peru',
        'Trinadad&Tobago'
    ]

    europa = [
        'England',
        'Germany',
        'France',
        'Italy',
        'Portugal',
        'Ireland',
        'Scotland',
        'Greece',
        'Holand-Netherlands',
        'Hungary',
        'Poland',
        'Yugoslavia'
    ]

    asia = [
        'India',
        'China',
        'Japan',
        'Philippines',
        'Vietnam',
        'Taiwan',
        'Hong',
        'Thailand',
        'Cambodia',
        'Laos',
        'Iran'
    ]

    if pais in america_norte:
        return 'North America'

    elif pais in latinoamerica:
        return 'Latin America'

    elif pais in europa:
        return 'Europe'

    elif pais in asia:
        return 'Asia'

    elif pais == 'Unknown':
        return 'Unknown'

    else:
        return 'Other'

In [8]:
df['continent'] = df['native-country'].apply(agrupar_continente)

df.drop(columns=['native-country'], inplace=True)

In [9]:
X = df.drop('income', axis=1)
y = df['income']

Ahora creamos una variable binaria que indica si el individuo presenta o no ganancias de capital.

In [10]:
df['has_capital_gain'] = (
    df['capital-gain'] > 0
).astype(int)

una vez hemos hecha esta fase de transformaciones podemos pasar a hacer los 2 tipos de encoding 

podriamos aplicar numerical encoding para educational num pero no es necesario ya que ya es numerica 

en vez de eso aplicaremos binary encoding a gender y podemos aplicar one hot encoding para el resto de variables 

In [11]:
#ia
X = df.drop('income', axis=1)
y = df['income']

In [12]:
#ia
X['gender'] = X['gender'].map({
    'Male': 1,
    'Female': 0
})

ya esta el binary encoding pasamos al one hot encoding 

In [13]:
X.select_dtypes(include='object').columns

C:\Users\Usuario\AppData\Local\Temp\ipykernel_18072\2418744409.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(include='object').columns


Index(['workclass', 'marital-status', 'occupation', 'relationship', 'race',
       'continent'],
      dtype='str')

In [14]:
X.columns.tolist()

['age',
 'workclass',
 'fnlwgt',
 'educational-num',
 'marital-status',
 'occupation',
 'relationship',
 'race',
 'gender',
 'capital-gain',
 'capital-loss',
 'hours-per-week',
 'continent',
 'has_capital_gain']

In [15]:
# One Hot Encoding para las variables categóricas restantes

X = pd.get_dummies(
    X,
    columns=[
        'workclass',
        'marital-status',
        'occupation',
        'relationship',
        'race',
        'continent'
    ],
    drop_first=True
)

X.head()

,age,fnlwgt,educational-num,gender,capital-gain,capital-loss,hours-per-week,has_capital_gain,workclass_Local-gov,workclass_Never-worked,...,relationship_Wife,race_Asian-Pac-Islander,race_Black,race_Other,race_White,continent_Europe,continent_Latin America,continent_North America,continent_Other,continent_Unknown
0,25,226802,7,1,0,0,40,0,False,False,...,False,False,True,False,False,False,False,True,False,False
1,38,89814,9,1,0,0,50,0,False,False,...,False,False,False,False,True,False,False,True,False,False
2,28,336951,12,1,0,0,40,0,True,False,...,False,False,False,False,True,False,False,True,False,False
3,44,160323,10,1,7688,0,40,1,False,False,...,False,False,True,False,False,False,False,True,False,False
4,18,103497,10,0,0,0,30,0,False,False,...,False,False,False,False,True,False,False,True,False,False


In [16]:
#ia
X.to_csv('X_procesado.csv', index=False)
y.to_csv('y_procesado.csv', index=False)

una vez ya estan hechos los encodings guardo los dtos transformados para poder usarlos en la parte de modelado 